# Kompletny przewodnik: Matplotlib + Seaborn

Przewodnik referencyjny (Python 3.10+, seaborn ≥ 0.13, matplotlib ≥ 3.7, pandas). Każdy blok kodu jest samodzielny po wykonaniu sekcji **0** (importy + dane). Kod jest celowo „inline" (bez opakowywania w funkcje), żeby łatwo go kopiować do notebooków.

**Konwencja w tabelach parametrów:** ✅ = wymagany, ⬜ = opcjonalny.

## Spis treści

0. Setup, dane przykładowe
1. Matplotlib: anatomia i formatowanie (osie, ticki, legendy, linie, cieniowanie, adnotacje)
2. Kolory: palety i colormapy
3. Wykresy liniowe i obszarowe
4. Wykresy słupkowe (w tym formatowanie wybranych słupków)
5. Pie / donut
6. Rozkłady: histogram, KDE, ECDF, rug, displot (następca distplot)
7. Wykresy kategoryczne: box, boxen, violin, strip, swarm, point, count, catplot
8. Zależności: scatter, relplot, lmplot, regplot, residplot
9. Wykresy dwuwymiarowe: jointplot, pairplot, hexbin, 2D hist/KDE
10. Macierze: heatmap, clustermap
11. Ridge plot (joyplot)
12. Layouty: subplots, mosaic, FacetGrid, PairGrid, JointGrid, dual axis
13. Wyróżnianie elementów: linie, cieniowanie, adnotacje (zbiorczo)
14. Zapis, wydajność, pułapki
15. Ściągawka: jaki wykres do jakiego pytania

## 0. Setup i dane przykładowe

In [ ]:
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker
import matplotlib.dates as mdates
from matplotlib.patches import Rectangle
import seaborn as sns

print(sns.__version__, mpl.__version__)

### Motyw globalny (`sns.set_theme`)

| Parametr | Wartości | Opis |
|---|---|---|
| `context` ⬜ | `"paper"`, `"notebook"`, `"talk"`, `"poster"` | Skaluje rozmiary fontów i linii (`talk` do prezentacji) |
| `style` ⬜ | `"darkgrid"`, `"whitegrid"`, `"dark"`, `"white"`, `"ticks"` | Tło, siatka, ramki |
| `palette` ⬜ | nazwa palety / lista kolorów | Domyślna paleta jakościowa |
| `font` ⬜ | np. `"DejaVu Sans"`, `"Arial"` | Rodzina fontu |
| `font_scale` ⬜ | float (1.0) | Mnożnik wielkości fontów |
| `rc` ⬜ | dict | Nadpisanie dowolnych `rcParams` matplotlib |

In [ ]:
sns.set_theme(
    context="notebook",
    style="whitegrid",
    palette="deep",
    font_scale=1.0,
    rc={
        "figure.dpi": 110,
        "axes.spines.top": False,      # usuwa górną i prawą ramkę (czystszy wygląd)
        "axes.spines.right": False,
        "axes.titleweight": "bold",
        "axes.titlesize": 13,
        "axes.labelsize": 11,
        "legend.frameon": False,
        "figure.autolayout": False,
    },
)

# Tymczasowa zmiana stylu tylko dla fragmentu kodu:
with sns.axes_style("ticks"):
    fig, ax = plt.subplots(figsize=(4, 2))
    ax.plot([1, 2, 3], [1, 4, 9])
    plt.show()

# Wybrane rcParams matplotlib, które warto znać:
# plt.rcParams["figure.figsize"] = (9, 5)
# plt.rcParams["savefig.dpi"] = 300
# plt.rcParams["font.family"] = "sans-serif"
# plt.rcParams["axes.grid"] = True
# plt.rcParams["lines.linewidth"] = 2
# plt.rcParams["axes.prop_cycle"] = mpl.cycler(color=["#1f77b4", "#ff7f0e", "#2ca02c"])

### Dane przykładowe: zbiór „sprzedaż", zawiera wszystkie typy danych

Kolumny: data/czas, kategoria nominalna (`region`, `channel`), kategoria uporządkowana (`segment`), liczba całkowita (`customer_age`, `units`), ciągłe (`income`, `unit_price`, `discount`, `revenue`, `delivery_days`), porządkowa 1–5 (`satisfaction`), logiczna (`is_returning`), tekst/ID (`order_id`), braki danych (`delivery_days`).

In [ ]:
rng = np.random.default_rng(42)
n = 1500

df = pd.DataFrame({
    "order_id": [f"ORD-{i:05d}" for i in range(n)],
    "order_date": pd.Timestamp("2023-01-01") + pd.to_timedelta(rng.integers(0, 730, n), unit="D"),
    "region": rng.choice(["Północ", "Południe", "Wschód", "Zachód", "Centrum"], n, p=[.2, .15, .2, .2, .25]),
    "channel": rng.choice(["Online", "Sklep", "Telefon"], n, p=[.55, .35, .10]),
    "segment": pd.Categorical(
        rng.choice(["Basic", "Standard", "Premium"], n, p=[.45, .40, .15]),
        categories=["Basic", "Standard", "Premium"], ordered=True),
    "customer_age": np.clip(rng.normal(40, 12, n), 18, 80).round().astype(int),
    "is_returning": rng.random(n) < 0.4,
})

seg_mult = df["segment"].astype(str).map({"Basic": 1.0, "Standard": 1.6, "Premium": 2.8}).to_numpy()
df["income"] = (rng.lognormal(8.2, 0.35, n) * (1 + (df["customer_age"] - 18) / 120)).round(0)
df["units"] = rng.poisson(3, n) + 1
df["unit_price"] = (rng.gamma(6, 12, n) * seg_mult).round(2)
df["discount"] = np.clip(rng.beta(2, 12, n), 0, 0.5).round(3)
df["revenue"] = (df["units"] * df["unit_price"] * (1 - df["discount"])).round(2)
df["delivery_days"] = (rng.gamma(3, 1.2, n) + (df["channel"] == "Online").to_numpy() * 1.5).round(1)
df["satisfaction"] = np.clip(
    np.round(4.2 - 0.15 * df["delivery_days"] + 0.3 * (df["segment"] == "Premium").to_numpy() + rng.normal(0, 0.8, n)),
    1, 5).astype(int)

# braki danych i kolumny pochodne
df.loc[rng.choice(n, 40, replace=False), "delivery_days"] = np.nan
df["month"] = df["order_date"].dt.to_period("M").dt.to_timestamp()
df["year"] = df["order_date"].dt.year
df["weekday"] = df["order_date"].dt.day_name()
df["returning_int"] = df["is_returning"].astype(int)   # 0/1 do regresji logistycznej

# agregat miesięczny (do wykresów liniowych)
df_month = (df.groupby(["month", "channel"], as_index=False)
              .agg(revenue=("revenue", "sum"), orders=("order_id", "size")))

# dane „szerokie" (do heatmap)
corr = df.select_dtypes("number").drop(columns=["year", "returning_int"]).corr()
pivot = df.pivot_table(index="region", columns="segment", values="revenue",
                       aggfunc="mean", observed=True)

df.head()

> Dane z Polars: `df_pl.to_pandas()` przed przekazaniem do seaborn. Seaborn 0.13 pracuje wewnętrznie na pandas, a konwersja jest tania względem czasu samego rysowania.

## 1. Matplotlib: anatomia i formatowanie

**Model obiektowy:** `Figure` (całe płótno) → `Axes` (pojedynczy wykres: osie, dane, tytuł) → elementy (`Line2D`, `Patch`, `Text`, `Axis`). Zawsze używaj stylu obiektowego (`fig, ax = plt.subplots()`), a nie `plt.plot()`. Daje kontrolę i działa przy wielu osiach.

**Dwa rodzaje funkcji seaborn:**

| Typ | Przykłady | Cechy |
|---|---|---|
| *axes-level* | `scatterplot`, `lineplot`, `histplot`, `kdeplot`, `boxplot`, `barplot`, `heatmap`… | Rysują na istniejącym `ax=`, zwracają `Axes`, łatwo łączyć w subplots |
| *figure-level* | `relplot`, `displot`, `catplot`, `lmplot`, `jointplot`, `pairplot`, `clustermap` | Tworzą własną figurę, mają `col=`/`row=`, zwracają `FacetGrid`/`JointGrid`; **nie przyjmują `ax=`**, rozmiar przez `height=` i `aspect=` |

### 1.1. Pełny przykład formatowania jednego wykresu

In [ ]:
monthly = df.groupby("month")["revenue"].sum()

fig, ax = plt.subplots(figsize=(10, 4.5), constrained_layout=True)

ax.plot(monthly.index, monthly.values, color="#1f77b4", linewidth=2.2,
        marker="o", markersize=6, markerfacecolor="white", markeredgewidth=2,
        linestyle="-", label="Przychód miesięczny")

# --- tytuł i etykiety
ax.set_title("Przychód miesięczny", loc="left", fontsize=14, fontweight="bold", pad=12)
ax.set_xlabel("Miesiąc", fontsize=11, labelpad=8)
ax.set_ylabel("Przychód [PLN]", fontsize=11)

# --- zakresy osi
ax.set_ylim(0, monthly.max() * 1.15)
ax.set_xlim(monthly.index.min() - pd.Timedelta(days=15), monthly.index.max() + pd.Timedelta(days=15))

# --- ticki: lokator + formatter
ax.xaxis.set_major_locator(mdates.MonthLocator(interval=3))
ax.xaxis.set_major_formatter(mdates.DateFormatter("%b %Y"))
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
ax.tick_params(axis="x", rotation=45, labelsize=9)
ax.tick_params(axis="y", labelsize=9, length=0)

# --- linie referencyjne
ax.axhline(monthly.mean(), color="crimson", linestyle="--", linewidth=1.4, label=f"Średnia: {monthly.mean():,.0f}")
ax.axvline(pd.Timestamp("2024-01-01"), color="gray", linestyle=":", linewidth=1.2)

# --- cieniowanie
ax.axvspan(pd.Timestamp("2023-11-01"), pd.Timestamp("2024-01-31"), color="orange", alpha=0.15, label="Sezon świąteczny")
ax.axhspan(monthly.mean() * 0.9, monthly.mean() * 1.1, color="crimson", alpha=0.08)  # pas ±10% wokół średniej
ax.fill_between(monthly.index, monthly.values, color="#1f77b4", alpha=0.10)           # wypełnienie pod linią

# --- adnotacja ze strzałką
peak = monthly.idxmax()
ax.annotate(f"Max: {monthly.max():,.0f}", xy=(peak, monthly.max()),
            xytext=(peak, monthly.max() * 1.08), ha="center", fontsize=10,
            arrowprops=dict(arrowstyle="->", color="black", lw=1.2))

# --- siatka, ramki, legenda
ax.grid(True, which="major", axis="y", linestyle="-", alpha=0.3)
ax.grid(False, axis="x")
ax.spines[["top", "right"]].set_visible(False)
ax.legend(loc="upper left", ncol=1, frameon=False, fontsize=9, title="Legenda", title_fontsize=10)

plt.show()

### 1.2. Kluczowe metody Axes (ściągawka)

| Cel | Metoda | Ważne parametry |
|---|---|---|
| Tytuł | `ax.set_title(label)` ✅ | `loc` (`left/center/right`), `fontsize`, `fontweight`, `pad`, `color` |
| Etykiety osi | `ax.set_xlabel(label)` ✅ | `fontsize`, `labelpad`, `color`, `rotation` |
| Zakres | `ax.set_xlim(left, right)` | `ax.set_ylim(bottom, top)`; `ax.invert_yaxis()` |
| Skala | `ax.set_xscale("log")` | `"log"`, `"symlog"`, `"logit"`; `base=`, `linthresh=` |
| Ticki | `ax.set_xticks(positions)` | `ax.set_xticklabels(labels, rotation=, ha=)` (po `set_xticks`!) |
| Ticki (styl) | `ax.tick_params()` | `axis`, `which`, `labelsize`, `rotation`, `length`, `width`, `direction`, `colors`, `pad` |
| Siatka | `ax.grid()` | `visible`, `which` (`major/minor/both`), `axis`, `linestyle`, `alpha`, `color` |
| Ramki | `ax.spines["top"].set_visible(False)` | `set_linewidth`, `set_color`, `set_position(("outward", 10))` |
| Legenda | `ax.legend()` | `loc`, `bbox_to_anchor`, `ncol`, `frameon`, `title`, `fontsize`, `handles`, `labels` |
| Tekst | `ax.text(x, y, s)` ✅ | `ha`, `va`, `fontsize`, `color`, `transform=ax.transAxes` (współrzędne 0–1), `bbox=dict(...)` |
| Adnotacja | `ax.annotate(text, xy)` ✅ | `xytext`, `arrowprops`, `textcoords`, `ha`, `va` |
| Proporcje | `ax.set_aspect("equal")` | `"equal"`, `"auto"`, liczba |
| Margines | `ax.margins(x=0.05, y=0.1)` | |

**Pozycjonowanie legendy poza wykresem:**

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
for ch in df["channel"].unique():
    s = df[df["channel"] == ch].groupby("month")["revenue"].sum()
    ax.plot(s.index, s.values, label=ch)
ax.legend(title="Kanał", loc="upper left", bbox_to_anchor=(1.01, 1), borderaxespad=0)  # na prawo od osi
plt.tight_layout()
plt.show()

### 1.3. Formattery i lokatory osi

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 3.8), constrained_layout=True)
s = df.groupby("region")["revenue"].sum().sort_values(ascending=False)

# a) tysiące / miliony przez FuncFormatter
axes[0].bar(s.index, s.values, color="#4c72b0")
axes[0].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, pos: f"{v/1e3:,.0f} tys."))
axes[0].set_title("FuncFormatter (tys.)")

# b) procenty (PercentFormatter: xmax = wartość odpowiadająca 100%)
axes[1].bar(s.index, s.values / s.sum(), color="#55a868")
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(xmax=1, decimals=0))
axes[1].set_title("PercentFormatter")

# c) lokatory: co ile ticki główne i pomocnicze
x = np.linspace(0, 100, 200)
axes[2].plot(x, np.sqrt(x))
axes[2].xaxis.set_major_locator(mticker.MultipleLocator(20))
axes[2].xaxis.set_minor_locator(mticker.MultipleLocator(5))
axes[2].yaxis.set_major_locator(mticker.MaxNLocator(nbins=5, integer=True))
axes[2].grid(which="minor", alpha=0.2)
axes[2].set_title("Locators")
plt.show()

Daty: `mdates.DayLocator`, `WeekdayLocator`, `MonthLocator(interval=)`, `YearLocator`; formattery: `DateFormatter("%Y-%m")`, `ConciseDateFormatter(locator)` (najczystszy, automatyczny).

### 1.4. Układ wielu wykresów

In [ ]:
# a) siatka subplots z współdzielonymi osiami
fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True, sharey=False, constrained_layout=True)
sns.histplot(df, x="revenue", ax=axes[0, 0])
sns.boxplot(df, x="revenue", ax=axes[0, 1])
sns.ecdfplot(df, x="revenue", ax=axes[1, 0])
sns.kdeplot(df, x="revenue", fill=True, ax=axes[1, 1])
fig.suptitle("Ten sam rozkład, cztery ujęcia", fontsize=14, fontweight="bold")
plt.show()

# b) mozaika (elastyczny layout o nierównych polach)
fig, axd = plt.subplot_mosaic(
    [["scatter", "scatter", "box"],
     ["scatter", "scatter", "hist"]],
    figsize=(11, 6), constrained_layout=True,
    # width_ratios=[2, 2, 1], height_ratios=[1, 1]
)
sns.scatterplot(df.sample(400, random_state=1), x="unit_price", y="revenue", hue="segment", ax=axd["scatter"])
sns.boxplot(df, x="segment", y="revenue", hue="segment", legend=False, ax=axd["box"])
sns.histplot(df, x="revenue", ax=axd["hist"])
plt.show()

| Parametr `plt.subplots` | Opis |
|---|---|
| `nrows`, `ncols` ⬜ | Liczba wierszy/kolumn |
| `figsize` ⬜ | Rozmiar w calach (szer., wys.) |
| `sharex`, `sharey` ⬜ | `True`, `"row"`, `"col"`; ułatwia porównanie skal |
| `constrained_layout` ⬜ | Automatyczne rozmieszczenie (lepsze niż `tight_layout`; nie łącz obu) |
| `gridspec_kw` ⬜ | np. `{"height_ratios": [3, 1], "hspace": 0.05}` |

## 2. Kolory: palety i colormapy

| Rodzaj danych | Typ palety | Przykłady |
|---|---|---|
| Kategorie nominalne | jakościowa (*qualitative*) | `"deep"`, `"Set2"`, `"tab10"`, `"colorblind"`, `"pastel"`, `"muted"` |
| Wartości uporządkowane / ciągłe | sekwencyjna (*sequential*) | `"viridis"`, `"rocket"`, `"mako"`, `"Blues"`, `"flare"`, `"crest"` |
| Wartości z neutralnym środkiem (korelacje, odchylenia) | rozbieżna (*diverging*) | `"coolwarm"`, `"vlag"`, `"RdBu_r"`, `"icefire"`, `"Spectral"` |

Dodaj `_r`, aby odwrócić (np. `"viridis_r"`).

In [ ]:
pals = {
    "deep (kat.)": sns.color_palette("deep", 8),
    "Set2 (kat.)": sns.color_palette("Set2", 8),
    "colorblind": sns.color_palette("colorblind", 8),
    "rocket (sekw.)": sns.color_palette("rocket", 8),
    "crest (sekw.)": sns.color_palette("crest", 8),
    "vlag (rozb.)": sns.color_palette("vlag", 8),
    "light_palette": sns.light_palette("seagreen", 8),
    "dark_palette": sns.dark_palette("#69d", 8),
    "cubehelix": sns.cubehelix_palette(8, start=.5, rot=-.75),
    "diverging_palette": sns.diverging_palette(220, 20, n=8),
}
fig, axes = plt.subplots(len(pals), 1, figsize=(7, 0.45 * len(pals)))
for ax, (name, pal) in zip(axes, pals.items()):
    ax.bar(range(len(pal)), 1, color=pal, width=1)
    ax.set_axis_off()
    ax.set_title(name, loc="left", fontsize=9, pad=2)
plt.show()

**Sposoby przekazywania kolorów w seaborn:**

In [ ]:
sns.color_palette("Set2")                                 # paleta nazwana
sns.color_palette(["#e63946", "#457b9d", "#2a9d8f"])      # własna lista
palette_map = {"Online": "#1f77b4", "Sklep": "#ff7f0e", "Telefon": "#7f7f7f"}   # słownik: stałe kolory per kategoria (zalecane!)
sns.set_palette("Set2")                                   # globalnie
cmap_custom = mpl.colors.LinearSegmentedColormap.from_list("moj", ["#ffffff", "#264653"])  # własna colormapa ciągła

> **Dobra praktyka:** używaj słownika `{kategoria: kolor}` konsekwentnie w całym raporcie. Ten sam kanał ma zawsze ten sam kolor.

## 3. Wykresy liniowe i obszarowe

**Kiedy:** szeregi czasowe, trendy, zmiana wartości wzdłuż uporządkowanej osi X.
**Interpretacja:** nachylenie = tempo zmiany; sezonowość = powtarzające się wzorce; pasmo wokół linii w `lineplot` = niepewność estymatora (domyślnie 95% przedział ufności średniej z bootstrapu), a **nie** zakres danych.
**Uwaga:** nie łącz linią punktów, między którymi nie ma logicznej ciągłości (np. kategorii nominalnych).

### 3.1. `sns.lineplot`: parametry

| Parametr | Opis |
|---|---|
| `data`, `x`, `y` ✅ | Dane i kolumny (bez `y` rysuje po indeksie dla wide-form) |
| `hue`, `style`, `size` ⬜ | Zmienna kodowana kolorem / stylem linii (`markers=True`, `dashes`) / grubością (`sizes`) |
| `units` ⬜ | Identyfikator jednostki; z `estimator=None` rysuje osobną linię na jednostkę |
| `estimator` ⬜ | Agregacja przy wielu y na x: `"mean"` (domyślnie), `"median"`, `"sum"`, `None` |
| `errorbar` ⬜ | `("ci", 95)`, `"sd"`, `"se"`, `("pi", 90)`, `None` (bez pasma, szybciej) |
| `err_style` ⬜ | `"band"` lub `"bars"` |
| `n_boot`, `seed` ⬜ | Parametry bootstrapu |
| `palette`, `hue_order`, `hue_norm` ⬜ | Kolory i kolejność |
| `markers`, `dashes` ⬜ | `True/False`, lista lub słownik |
| `sort` ⬜ | Sortowanie po x (domyślnie `True`) |
| `ax` ⬜ | Docelowe osie |
| `**kwargs` ⬜ | `linewidth`, `alpha`, `linestyle`, `marker`, `markersize`, `markerfacecolor` |

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 8), constrained_layout=True)

# a) wiele serii: hue + style + markery
sns.lineplot(df_month, x="month", y="revenue", hue="channel", style="channel",
             markers=True, dashes=False, linewidth=2, markersize=7,
             palette={"Online": "#1f77b4", "Sklep": "#ff7f0e", "Telefon": "#7f7f7f"}, ax=axes[0, 0])
axes[0, 0].set_title("hue + style + markers")

# b) dane surowe: średnia + 95% CI (bootstrap)
sns.lineplot(df, x="month", y="revenue", errorbar=("ci", 95), color="#2a9d8f", ax=axes[0, 1])
axes[0, 1].set_title("Średnia + 95% CI")

# c) odchylenie standardowe zamiast CI + inny estymator
sns.lineplot(df, x="month", y="revenue", estimator="median", errorbar=("pi", 50),
             color="#e76f51", ax=axes[1, 0])
axes[1, 0].set_title("Mediana + rozstęp 50% obserwacji (pi)")

# d) err_style="bars"
sns.lineplot(df, x="month", y="revenue", errorbar="se", err_style="bars",
             err_kws={"capsize": 3}, marker="o", color="#264653", ax=axes[1, 1])
axes[1, 1].set_title("Błąd standardowy (słupki błędów)")

for ax in axes.flat:
    ax.tick_params(axis="x", rotation=45)
plt.show()

### 3.2. Średnia krocząca, pasmo zmienności, wyróżnienie okresu

In [ ]:
s = df.groupby("month")["revenue"].sum()
roll = s.rolling(3, center=True).mean()
std = s.rolling(3, center=True).std()

fig, ax = plt.subplots(figsize=(10, 4.5), constrained_layout=True)
ax.plot(s.index, s, color="lightgray", marker="o", label="Dane miesięczne")
ax.plot(roll.index, roll, color="#1f77b4", linewidth=2.5, label="Średnia krocząca (3 m-ce)")
ax.fill_between(roll.index, roll - std, roll + std, color="#1f77b4", alpha=0.2, label="±1 σ (okno 3 m-ce)")
ax.axhline(s.mean(), color="crimson", linestyle="--", label="Średnia całkowita")
ax.scatter(s.idxmax(), s.max(), s=150, color="gold", edgecolor="black", zorder=5, label="Maksimum")
ax.fill_between(s.index, s, s.mean(), where=(s > s.mean()), color="green", alpha=0.15, interpolate=True)
ax.fill_between(s.index, s, s.mean(), where=(s <= s.mean()), color="red", alpha=0.15, interpolate=True)
ax.legend(ncol=2, loc="upper left")
ax.yaxis.set_major_formatter(mticker.StrMethodFormatter("{x:,.0f}"))
plt.show()

### 3.3. Wykres obszarowy (stacked area)

**Kiedy:** skład całości w czasie. **Uwaga:** poza dolną warstwą trudno porównywać wartości, bo mają różne linie bazowe.

In [ ]:
wide = df_month.pivot(index="month", columns="channel", values="revenue").fillna(0)

fig, axes = plt.subplots(1, 2, figsize=(14, 4.5), constrained_layout=True)
axes[0].stackplot(wide.index, wide.T.values, labels=wide.columns,
                  colors=sns.color_palette("Set2", 3), alpha=0.9)
axes[0].set_title("Stacked area (wartości)")
axes[0].legend(loc="upper left")

share = wide.div(wide.sum(axis=1), axis=0)
axes[1].stackplot(share.index, share.T.values, labels=share.columns, colors=sns.color_palette("Set2", 3))
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(1))
axes[1].set_title("Udział kanałów (100%)")
plt.show()

### 3.4. Wykres z błędami (`errorbar`) i step

In [ ]:
g = df.groupby("region")["revenue"].agg(["mean", "std", "count"])
g["sem"] = g["std"] / np.sqrt(g["count"])
fig, ax = plt.subplots(figsize=(7, 4))
ax.errorbar(g.index, g["mean"], yerr=1.96 * g["sem"], fmt="o", capsize=5, color="#264653", ecolor="#e76f51", elinewidth=2, markersize=8)
ax.set_title("Średnia ± 95% CI (przybliżone)")
plt.show()

## 4. Wykresy słupkowe

**Kiedy:** porównanie wartości (lub agregatów) między kategoriami. **Zawsze zaczynaj oś Y od 0**: długość słupka koduje wartość.
**Interpretacja `sns.barplot`:** wysokość = estymator (domyślnie średnia), „wąsy" = przedział niepewności estymatora. **Nie jest to rozkład** (do tego box/violin).

### 4.1. `sns.barplot`: parametry

| Parametr | Opis |
|---|---|
| `data`, `x`, `y` ✅ | Dane i zmienne (kategoria + wartość; zamień osie → słupki poziome) |
| `hue` ⬜ | Podział na podgrupy (słupki grupowane); `dodge=False` łączy je w jeden słupek |
| `estimator` ⬜ | `"mean"` (domyślnie), `"sum"`, `"median"`, `"count"`, funkcja |
| `errorbar` ⬜ | `("ci", 95)`, `"sd"`, `"se"`, `None` |
| `capsize` ⬜ | Szerokość kresek na końcach wąsów |
| `order`, `hue_order` ⬜ | Kolejność kategorii |
| `orient` ⬜ | `"v"`/`"h"` (zwykle wykrywane) |
| `width` ⬜ | Szerokość słupka (0–1) |
| `gap` ⬜ | Odstęp między słupkami w grupie (0.13+) |
| `saturation` ⬜ | Nasycenie kolorów (1 = pełne) |
| `fill` ⬜ | `False` = same obrysy |
| `palette`, `color` ⬜ | Paleta (dla `hue`) lub jeden kolor |
| `err_kws` ⬜ | np. `{"linewidth": 1.5, "color": "black"}` |
| `ax` ⬜ | Osie |
| `**kwargs` ⬜ | `edgecolor`, `linewidth`, `alpha`, `hatch` |

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 9), constrained_layout=True)

# a) średnia + 95% CI, wyraźna kolejność
order = df.groupby("region")["revenue"].mean().sort_values(ascending=False).index
sns.barplot(df, x="region", y="revenue", order=order, hue="region", legend=False,
            palette="viridis", errorbar=("ci", 95), capsize=0.15,
            err_kws={"linewidth": 1.5}, ax=axes[0, 0])
axes[0, 0].set_title("Średnia + 95% CI (posortowane)")

# b) suma zamiast średniej, bez wąsów, etykiety wartości
sns.barplot(df, x="region", y="revenue", estimator="sum", errorbar=None,
            order=order, color="#4c72b0", ax=axes[0, 1])
axes[0, 1].bar_label(axes[0, 1].containers[0], fmt="{:,.0f}", padding=3, fontsize=9)
axes[0, 1].set_title("Suma + etykiety (bar_label)")
axes[0, 1].yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v/1e3:,.0f}k"))

# c) poziome słupki, hue (grupowane), odstęp w grupie
sns.barplot(df, y="region", x="revenue", hue="segment", estimator="mean", errorbar=None,
            order=order, palette="rocket", gap=0.1, width=0.8, edgecolor="white", ax=axes[0, 2])
axes[0, 2].set_title("Poziome, grupowane (hue)")
axes[0, 2].legend(title="Segment", loc="lower right")

# d) wyróżnienie jednego słupka (słownik kolorów)
tot = df.groupby("region", as_index=False)["revenue"].sum().sort_values("revenue", ascending=False)
cmap_hl = {r: ("#d62728" if r == tot.iloc[0]["region"] else "#b8c0cc") for r in tot["region"]}
sns.barplot(tot, x="region", y="revenue", hue="region", palette=cmap_hl, legend=False, ax=axes[1, 0])
axes[1, 0].bar_label(axes[1, 0].containers[0], fmt="{:,.0f}", padding=2)
axes[1, 0].axhline(tot["revenue"].mean(), color="black", linestyle="--", linewidth=1)
axes[1, 0].text(4.4, tot["revenue"].mean(), " średnia", va="bottom", ha="right", fontsize=9)
axes[1, 0].set_title("Wyróżniony najlepszy region")

# e) formatowanie pojedynczych słupków PO narysowaniu (patches)
sns.barplot(tot, x="region", y="revenue", color="#8da0cb", ax=axes[1, 1])
for bar, val in zip(axes[1, 1].patches, tot["revenue"]):
    if val < tot["revenue"].mean():                       # słupki poniżej średniej
        bar.set_facecolor("#f4a261")
        bar.set_hatch("//")
        bar.set_edgecolor("black")
axes[1, 1].set_title("Warunkowe formatowanie (patches)")

# f) wypełnienie i hatch jako obrys
sns.barplot(tot, x="region", y="revenue", fill=False, edgecolor="black", linewidth=2,
            hatch="..", errorbar=None, ax=axes[1, 2])
axes[1, 2].set_title("fill=False + hatch")

for ax in axes.flat:
    ax.set_xlabel("")
plt.show()

### 4.2. Słupki skumulowane i 100%

Seaborn nie ma „stacked bar": użyj pandas `.plot` (matplotlib) albo `sns.histplot(multiple="stack")`.

In [ ]:
ct = pd.crosstab(df["region"], df["segment"])           # liczby
ct_pct = ct.div(ct.sum(axis=1), axis=0)                 # udziały wiersza

fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), constrained_layout=True)

ct.plot(kind="bar", stacked=True, ax=axes[0], colormap="viridis", edgecolor="white", width=0.8)
axes[0].set_title("Skumulowane (liczby)")

ct_pct.plot(kind="bar", stacked=True, ax=axes[1], colormap="viridis", edgecolor="white")
axes[1].yaxis.set_major_formatter(mticker.PercentFormatter(1))
for c in axes[1].containers:                            # etykiety w segmentach
    axes[1].bar_label(c, labels=[f"{v:.0%}" for v in c.datavalues], label_type="center", color="white", fontsize=9)
axes[1].set_title("100% (udział segmentów)")

# alternatywa w seaborn: histplot z multiple="fill" na kolumnie kategorycznej
sns.histplot(df, x="region", hue="segment", multiple="fill", stat="proportion",
             shrink=0.8, palette="viridis", ax=axes[2])
axes[2].set_title("histplot(multiple='fill')")
for ax in axes:
    ax.tick_params(axis="x", rotation=0)
plt.show()

### 4.3. Wykres lizakowy (lollipop) i „dumbbell": czytelniejsze niż słupki przy wielu kategoriach

In [ ]:
s = df.groupby("region")["revenue"].mean().sort_values()
fig, ax = plt.subplots(figsize=(7, 4))
ax.hlines(y=s.index, xmin=0, xmax=s.values, color="#8da0cb", linewidth=2)
ax.scatter(s.values, s.index, s=120, color="#264653", zorder=3)
ax.set_xlim(0)
ax.set_title("Lollipop: średni przychód / region")
plt.show()

## 5. Pie / donut

**Kiedy:** **wyłącznie** udział części w całości, przy ≤ 5 kategoriach, gdy różnice są duże. **Unikaj** przy wielu kategoriach i bliskich wartościach (oko słabo porównuje kąty), a słupki 100% są prawie zawsze lepsze.
**Interpretacja:** kąt/pole wycinka = udział w sumie.

| Parametr `ax.pie` | Opis |
|---|---|
| `x` ✅ | Wartości (automatycznie normalizowane do 100%) |
| `labels` ⬜ | Etykiety wycinków |
| `autopct` ⬜ | Format procentów: `"%1.1f%%"` lub funkcja |
| `colors` ⬜ | Lista kolorów |
| `explode` ⬜ | Wysunięcie wycinków, np. `[0.05, 0, 0]` |
| `startangle` ⬜ | Kąt startu (90 = od góry) |
| `counterclock` ⬜ | `False` = zgodnie z ruchem zegara |
| `wedgeprops` ⬜ | `{"width": 0.4}` (donut), `{"edgecolor": "white", "linewidth": 2}` |
| `pctdistance`, `labeldistance` ⬜ | Odległość procentów / etykiet od środka |
| `textprops` ⬜ | Styl tekstu `{"fontsize": 11}` |
| `shadow` ⬜ | Cień |

In [ ]:
vals = df.groupby("channel")["revenue"].sum().sort_values(ascending=False)
colors = sns.color_palette("pastel", len(vals))

fig, axes = plt.subplots(1, 2, figsize=(11, 5))

axes[0].pie(vals, labels=vals.index, autopct="%1.1f%%", startangle=90, counterclock=False,
            explode=[0.06] + [0] * (len(vals) - 1), colors=colors,
            wedgeprops={"edgecolor": "white", "linewidth": 2}, pctdistance=0.75,
            textprops={"fontsize": 11})
axes[0].set_title("Pie + explode")

wedges, texts, autotexts = axes[1].pie(
    vals, labels=vals.index, autopct="%1.0f%%", startangle=90, counterclock=False, colors=colors,
    wedgeprops={"width": 0.4, "edgecolor": "white", "linewidth": 2}, pctdistance=0.8)
axes[1].text(0, 0, f"{vals.sum()/1e3:,.0f} tys.\nPLN", ha="center", va="center", fontsize=14, fontweight="bold")
plt.setp(autotexts, color="black", fontsize=10)
axes[1].set_title("Donut z podsumowaniem w środku")
plt.show()

## 6. Rozkłady zmiennej ciągłej

### 6.1. Histogram: `sns.histplot`

**Kiedy:** kształt rozkładu (symetria, skośność, wielomodalność, wartości odstające). **Interpretacja:** wysokość = liczba (lub gęstość/proporcja) obserwacji w przedziale. Wygląd zależy od liczby przedziałów, więc zawsze sprawdź 2-3 ustawienia `bins`.

| Parametr | Opis |
|---|---|
| `data`, `x` (lub `y`) ✅ | Dane i zmienna (`y=` → pionowy histogram; `x`+`y` → histogram 2D) |
| `bins` ⬜ | Liczba (`30`), reguła (`"auto"`, `"fd"`, `"sturges"`, `"doane"`), lub lista krawędzi |
| `binwidth`, `binrange` ⬜ | Szerokość przedziału / zakres `(min, max)` |
| `discrete` ⬜ | `True` dla zmiennych całkowitych (słupki wycentrowane na wartościach) |
| `stat` ⬜ | `"count"`, `"frequency"`, `"probability"`, `"percent"`, `"density"` |
| `kde` ⬜ | Nakłada krzywą KDE (`kde_kws={"bw_adjust": ...}`) |
| `hue` ⬜ | Podział na grupy |
| `multiple` ⬜ | `"layer"`, `"stack"`, `"dodge"`, `"fill"` |
| `element` ⬜ | `"bars"`, `"step"`, `"poly"` |
| `fill` ⬜ | Wypełnienie |
| `cumulative` ⬜ | Dystrybuanta skumulowana |
| `common_norm` ⬜ | `False` = każda grupa normalizowana osobno (porównanie kształtów przy nierównych liczebnościach) |
| `log_scale` ⬜ | Skala log (osie X) |
| `shrink` ⬜ | Zwęża słupki |
| `cbar` ⬜ | Pasek kolorów (2D) |
| `palette`, `color`, `alpha`, `edgecolor`, `linewidth` ⬜ | Wygląd |

In [ ]:
fig, axes = plt.subplots(2, 4, figsize=(20, 8), constrained_layout=True)

sns.histplot(df, x="revenue", bins=40, color="#4c72b0", ax=axes[0, 0])
axes[0, 0].set_title("bins=40")

sns.histplot(df, x="revenue", bins="fd", stat="density", kde=True, color="#55a868", ax=axes[0, 1])
axes[0, 1].set_title("bins='fd', stat='density', kde=True")

sns.histplot(df, x="revenue", hue="segment", multiple="stack", palette="viridis", ax=axes[0, 2])
axes[0, 2].set_title("hue + multiple='stack'")

sns.histplot(df, x="revenue", hue="segment", element="step", stat="density", common_norm=False,
             fill=False, linewidth=2, palette="viridis", ax=axes[0, 3])
axes[0, 3].set_title("step, density, common_norm=False")

sns.histplot(df, x="revenue", log_scale=True, kde=True, color="#c44e52", ax=axes[1, 0])
axes[1, 0].set_title("log_scale=True (rozkład skośny)")

sns.histplot(df, x="satisfaction", discrete=True, shrink=0.8, stat="percent", color="#8172b3", ax=axes[1, 1])
axes[1, 1].set_title("discrete=True (zmienna porządkowa)")

sns.histplot(df, x="revenue", cumulative=True, stat="probability", element="step", fill=False, ax=axes[1, 2])
axes[1, 2].set_title("cumulative=True")

sns.histplot(df, x="unit_price", y="revenue", bins=40, cbar=True, cmap="mako", ax=axes[1, 3])
axes[1, 3].set_title("Histogram 2D")

# linie referencyjne na pierwszym wykresie
m, med = df["revenue"].mean(), df["revenue"].median()
axes[0, 0].axvline(m, color="red", linestyle="--", label=f"średnia {m:,.0f}")
axes[0, 0].axvline(med, color="black", linestyle=":", label=f"mediana {med:,.0f}")
axes[0, 0].legend()
plt.show()

**Interpretacja przykładu:** średnia > mediana ⇒ skośność prawostronna (ogon po prawej). Przy takim rozkładzie `log_scale=True` ujawnia strukturę, a do opisu lepsza jest mediana.

### 6.2. KDE: `sns.kdeplot`

**Kiedy:** gładki szacunek gęstości, szczególnie do porównania rozkładów między grupami. **Pułapki:** KDE „wygładza" i może pokazać gęstość poza zakresem danych (`cut=0` / `clip=`); przy małych próbach wprowadza w błąd; wysokość = gęstość, nie liczność.

| Parametr | Opis |
|---|---|
| `data`, `x` (lub `y`) ✅ | Dla `x`+`y` → KDE 2D (kontury) |
| `bw_adjust` ⬜ | Mnożnik szerokości jądra (<1 więcej szczegółów, >1 gładziej) |
| `bw_method` ⬜ | `"scott"`, `"silverman"` lub liczba |
| `fill` ⬜ | Wypełnienie pod krzywą |
| `hue` ⬜ | Grupy |
| `multiple` ⬜ | `"layer"`, `"stack"`, `"fill"` |
| `common_norm` ⬜ | Wspólna / osobna normalizacja grup |
| `cut` ⬜ | Jak daleko poza dane rozciągać krzywą (domyślnie 3, `0` = przycięta do danych) |
| `clip` ⬜ | Twardy zakres `(min, max)` |
| `cumulative` ⬜ | Skumulowana |
| `log_scale` ⬜ | Skala log |
| `levels`, `thresh` ⬜ | Dla 2D: liczba poziomów / próg pomijanej gęstości |
| `cbar` ⬜ | Pasek kolorów (2D) |
| `palette`, `color`, `alpha`, `linewidth` ⬜ | Wygląd |

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 8), constrained_layout=True)

sns.kdeplot(df, x="delivery_days", fill=True, color="#4c72b0", ax=axes[0, 0])
axes[0, 0].set_title("Podstawowy (fill=True)")

sns.kdeplot(df, x="delivery_days", hue="channel", fill=True, alpha=0.35, common_norm=False,
            palette="Set2", linewidth=2, ax=axes[0, 1])
axes[0, 1].set_title("hue, common_norm=False")

for bw, ls in [(0.3, ":"), (1, "-"), (2.5, "--")]:
    sns.kdeplot(df, x="delivery_days", bw_adjust=bw, linestyle=ls, label=f"bw_adjust={bw}", color="black", ax=axes[0, 2])
axes[0, 2].legend()
axes[0, 2].set_title("Wpływ bw_adjust")

sns.kdeplot(df, x="delivery_days", hue="channel", multiple="fill", palette="Set2", cut=0, ax=axes[1, 0])
axes[1, 0].set_title("multiple='fill', cut=0")

sns.kdeplot(df, x="unit_price", y="revenue", fill=True, levels=8, thresh=0.05, cmap="rocket_r", ax=axes[1, 1])
axes[1, 1].set_title("KDE 2D (fill, levels)")

sns.kdeplot(df, x="unit_price", y="revenue", hue="segment", levels=4, palette="viridis", ax=axes[1, 2])
axes[1, 2].set_title("KDE 2D + hue (kontury)")
plt.show()

### 6.3. `distplot` → `displot` (figure-level)

`sns.distplot` jest **przestarzały** (deprecated od 0.11, w nowszych wersjach usunięty). Zamiennik:

| Stare `distplot` | Nowe |
|---|---|
| `distplot(x)` | `histplot(x, kde=True, stat="density")` lub `displot(..., kind="hist", kde=True)` |
| `distplot(x, hist=False)` | `kdeplot(x)` |
| `distplot(x, rug=True)` | `displot(..., rug=True)` lub `rugplot` |

`displot` = figure-level: `kind` (`"hist"`, `"kde"`, `"ecdf"`), `col=`, `row=`, `col_wrap=`, `height=`, `aspect=`, `facet_kws=`, `rug=`.

In [ ]:
g = sns.displot(df, x="revenue", hue="segment", col="channel", kind="hist", stat="density",
                common_norm=False, element="step", kde=True, rug=True,
                rug_kws={"alpha": 0.2, "height": 0.03},
                palette="viridis", height=3.8, aspect=1.1, facet_kws={"sharey": False})
g.set_axis_labels("Przychód", "Gęstość")
g.set_titles("Kanał: {col_name}")
g.figure.suptitle("displot: hist + KDE + rug, panele per kanał", y=1.04, fontweight="bold")
plt.show()

### 6.4. ECDF: `sns.ecdfplot`

**Kiedy:** porównanie rozkładów **bez** wyboru binów/szerokości jądra; odczyt percentyli; porównanie grup, gdy KDE i histogram są nieczytelne.
**Interpretacja:** y = odsetek obserwacji ≤ x. Mediana = x przy y=0,5. Im bardziej krzywa „w prawo/dół", tym wyższe wartości. Przecinające się krzywe = rozkłady różnią się w różnych częściach zakresu.

| Parametr | Opis |
|---|---|
| `data`, `x` (lub `y`) ✅ | |
| `hue` ⬜ | Grupy |
| `stat` ⬜ | `"proportion"`, `"percent"`, `"count"` |
| `complementary` ⬜ | `True` = 1 − ECDF (odsetek obserwacji > x, tzw. krzywa przeżycia) |
| `weights` ⬜ | Wagi obserwacji |
| `log_scale` ⬜ | Skala log |
| `palette`, `linewidth`, `linestyle` ⬜ | Wygląd |

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.5), constrained_layout=True)

sns.ecdfplot(df, x="revenue", hue="segment", palette="viridis", linewidth=2, ax=axes[0])
for q in [0.5, 0.9]:                                                  # percentyle
    axes[0].axhline(q, color="gray", linestyle=":", linewidth=1)
axes[0].set_title("ECDF per segment (+ P50, P90)")

sns.ecdfplot(df, x="delivery_days", hue="channel", complementary=True, stat="percent", ax=axes[1])
axes[1].set_title("Complementary: % dostaw trwających > x dni")
axes[1].axvline(5, color="red", linestyle="--")

sns.ecdfplot(df, x="revenue", log_scale=True, stat="count", color="#c44e52", ax=axes[2])
axes[2].set_title("stat='count', log_scale")
plt.show()

### 6.5. Rug: `sns.rugplot`

**Kiedy:** dodatek do KDE/scatter: pokazuje **gdzie leżą pojedyncze obserwacje** (gęstość, luki, odstające).

| Parametr | Opis |
|---|---|
| `x` / `y` ✅ | Zmienna |
| `height` ⬜ | Wysokość kresek (ułamek osi, 0.025) |
| `expand_margins` ⬜ | Poszerza marginesy, żeby kreski nie zasłaniały danych |
| `hue` ⬜ | Grupy |
| `lw`, `alpha`, `color`, `clip_on` ⬜ | Wygląd |

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4))
sns.kdeplot(df, x="customer_age", fill=True, color="#4c72b0", ax=ax)
sns.rugplot(df, x="customer_age", height=0.05, alpha=0.15, color="black", ax=ax)
ax.set_title("KDE + rugplot")
plt.show()

## 7. Wykresy kategoryczne (rozkład w grupach)

### 7.1. Boxplot: `sns.boxplot`

**Kiedy:** porównanie rozkładów wielu grup w skondensowanej formie (mediana, rozrzut, outliery).
**Interpretacja:** linia w pudełku = **mediana**; pudełko = **IQR** (Q1–Q3, środkowe 50% danych); wąsy = ostatnie obserwacje w zakresie `whis × IQR` (domyślnie 1,5); punkty poza wąsami = potencjalne outliery. Przesunięcie mediany względem środka pudełka = skośność. Box **nie pokazuje wielomodalności** i liczebności grup.

| Parametr | Opis |
|---|---|
| `data`, `x`, `y` ✅ | Kategoria + zmienna ciągła (zamień osie → poziomo) |
| `hue` ⬜ | Podgrupy; `dodge`, `gap` sterują odstępem |
| `order`, `hue_order` ⬜ | Kolejność |
| `whis` ⬜ | Zasięg wąsów: `1.5` (domyślnie), `(5, 95)` = percentyle, `(0, 100)` = min–max |
| `notch` ⬜ | Wcięcie = przybliżony CI mediany (nie nakładające się wcięcia sugerują różnicę median) |
| `showmeans` ⬜ | Pokazuje średnią; `meanprops` styl znacznika |
| `showfliers`, `fliersize` ⬜ | Pokazywanie/rozmiar outlierów (`flierprops` pełny styl) |
| `width` ⬜ | Szerokość pudełka |
| `fill` ⬜ | `False` = same obrysy |
| `saturation`, `linewidth`, `linecolor`, `color`, `palette` ⬜ | Wygląd |
| `boxprops`, `whiskerprops`, `capprops`, `medianprops` ⬜ | Style elementów (dict, przekazywane do matplotlib) |
| `log_scale` ⬜ | Skala log |

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 9), constrained_layout=True)

sns.boxplot(df, x="segment", y="revenue", hue="segment", palette="viridis", legend=False, ax=axes[0, 0])
axes[0, 0].set_title("Podstawowy")

sns.boxplot(df, x="segment", y="revenue", hue="channel", palette="Set2", gap=0.1, width=0.7, ax=axes[0, 1])
axes[0, 1].set_title("hue (grupowany) + gap")

sns.boxplot(df, x="segment", y="revenue", hue="segment", legend=False, palette="pastel",
            showmeans=True,
            meanprops={"marker": "D", "markerfacecolor": "white", "markeredgecolor": "black", "markersize": 7},
            medianprops={"color": "black", "linewidth": 2},
            whiskerprops={"linewidth": 1.5}, capprops={"linewidth": 1.5},
            flierprops={"marker": "o", "markersize": 3, "alpha": 0.4, "markerfacecolor": "gray"},
            ax=axes[0, 2])
axes[0, 2].set_title("Pełne formatowanie elementów")

sns.boxplot(df, y="segment", x="revenue", hue="segment", legend=False, palette="mako",
            notch=True, whis=(5, 95), showfliers=False, ax=axes[1, 0])
axes[1, 0].set_title("Poziomy, notch, whis=(5,95), bez outlierów")

sns.boxplot(df, x="segment", y="revenue", fill=False, linewidth=2, color="#264653", log_scale=True, ax=axes[1, 1])
axes[1, 1].set_title("fill=False + log_scale")

# Boxplot + punkty (strip) = widać i podsumowanie, i dane
sns.boxplot(df, x="region", y="delivery_days", color="white", linewidth=1.5, showfliers=False, ax=axes[1, 2])
sns.stripplot(df.sample(500, random_state=1), x="region", y="delivery_days",
              size=3, alpha=0.5, jitter=0.25, color="#e76f51", ax=axes[1, 2])
axes[1, 2].set_title("Box + strip")
plt.show()

### 7.2. Boxenplot (letter-value plot): `sns.boxenplot`

**Kiedy:** **duże próby (> ~1 000 / grupę)**: pokazuje więcej kwantyli niż box, lepiej opisuje ogony. **Interpretacja:** środkowa linia = mediana; każda kolejna para „pudełek" = coraz głębsze kwantyle (kwartyle, oktyle, …), zawężające się ku ogonom.

Parametry: `k_depth` (`"tukey"`, `"proportion"`, `"trustworthy"`, `"full"` lub liczba poziomów), `width_method`, `width`, `linewidth`, `saturation`, `showfliers`, `hue`, `order`, `palette`.

In [ ]:
fig, ax = plt.subplots(figsize=(7, 4.5))
sns.boxenplot(df, x="segment", y="revenue", hue="segment", legend=False,
              palette="rocket", k_depth="trustworthy", linewidth=1, ax=ax)
ax.set_title("Boxenplot")
plt.show()

### 7.3. Violinplot: `sns.violinplot`

**Kiedy:** pełny kształt rozkładu w grupach (wielomodalność, skośność), gdy próby są wystarczająco duże (≥ 30–50 / grupę).
**Interpretacja:** szerokość „skrzypiec" = gęstość (KDE) w danej wartości; wewnątrz zwykle mini-box lub kwartyle. Uwaga: KDE wychodzi poza zakres danych, jeśli `cut > 0`.

| Parametr | Opis |
|---|---|
| `data`, `x`, `y` ✅ | Jak boxplot |
| `hue` ⬜ | Podgrupy |
| `split` ⬜ | `True` + `hue` o 2 poziomach: połówki skrzypiec obok siebie (bardzo dobre do porównania 2 grup) |
| `inner` ⬜ | `"box"`, `"quart"`, `"point"`, `"stick"`, `None`; `inner_kws` styl |
| `cut` ⬜ | Rozszerzenie poza dane (0 = przycięcie do zakresu) |
| `bw_adjust` ⬜ | Wygładzanie KDE |
| `density_norm` ⬜ | `"area"` (równe pola), `"count"` (szerokość ∝ liczebności), `"width"` (równe szerokości) |
| `common_norm` ⬜ | Wspólna normalizacja |
| `gap` ⬜ | Odstęp między skrzypcami w `hue` / przy `split` |
| `fill`, `linewidth`, `linecolor`, `saturation`, `palette`, `order` ⬜ | Wygląd |

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(14, 9), constrained_layout=True)

sns.violinplot(df, x="segment", y="revenue", hue="segment", legend=False, palette="viridis",
               inner="box", cut=0, ax=axes[0, 0])
axes[0, 0].set_title("inner='box', cut=0")

sns.violinplot(df, x="channel", y="delivery_days", hue="is_returning", split=True, inner="quart",
               gap=0.05, palette={True: "#2a9d8f", False: "#e76f51"}, ax=axes[0, 1])
axes[0, 1].set_title("split=True: stali vs nowi klienci")
axes[0, 1].legend(title="Powracający", loc="upper right")

sns.violinplot(df, x="region", y="customer_age", hue="region", legend=False, palette="Set2",
               density_norm="count", inner="stick", bw_adjust=0.8, ax=axes[1, 0])
axes[1, 0].set_title("density_norm='count', inner='stick'")

sns.violinplot(df, x="revenue", y="segment", hue="segment", legend=False, palette="mako",
               inner=None, cut=0, linewidth=0, ax=axes[1, 1])
sns.stripplot(df.sample(300, random_state=2), x="revenue", y="segment", size=2.5, color="black", alpha=0.4, ax=axes[1, 1])
axes[1, 1].set_title("Poziomy + punkty (inner=None)")
plt.show()

### 7.4. Strip i swarm: `sns.stripplot`, `sns.swarmplot`

**Kiedy:** **małe/średnie próby** (do ~ kilkuset punktów na grupę): pokazują każdą obserwację. Duże próby → overplotting, wybierz box/violin/boxen.
**Strip:** punkty z losowym jitterem (szybki, skaluje się). **Swarm:** punkty ułożone tak, by się nie nakładały (kształt przypomina violin, ale wolny). Gdy nie mieszczą się wszystkie punkty, seaborn ostrzega: zmniejsz `size` albo przejdź na `stripplot`. Swarm źle działa na zmiennych dyskretnych o kilku wartościach (np. skala 1–5), bo punkty układają się w pasy.

| Parametr | Opis |
|---|---|
| `data`, `x`, `y` ✅ | |
| `hue`, `dodge` ⬜ | Kolor i rozsunięcie podgrup |
| `jitter` ⬜ | (strip) `True` / float szerokość rozrzutu |
| `size` ⬜ | Rozmiar punktu |
| `alpha`, `edgecolor`, `linewidth` ⬜ | Przezroczystość i obrys |
| `native_scale` ⬜ | Pozycje wg wartości liczbowych osi kategorycznej |
| `warn_thresh` ⬜ | (swarm) próg ostrzeżenia o nakładaniu |
| `marker`, `palette`, `order` ⬜ | |

In [ ]:
sample = df.sample(300, random_state=3)
fig, axes = plt.subplots(1, 3, figsize=(17, 5), constrained_layout=True)

sns.stripplot(sample, x="segment", y="revenue", hue="channel", dodge=True, jitter=0.2,
              size=4, alpha=0.6, palette="Set2", ax=axes[0])
axes[0].set_title("stripplot: dodge + jitter")

sns.swarmplot(sample, x="segment", y="delivery_days", hue="segment", legend=False,
              size=2.2, palette="viridis", edgecolor="black", linewidth=0.2, ax=axes[1])
axes[1].set_title("swarmplot")

# swarm + średnia jako wyróżniony punkt
sns.swarmplot(sample, x="region", y="delivery_days", color="#b0b7c3", size=2, ax=axes[2])
sns.pointplot(sample, x="region", y="delivery_days", color="crimson", linestyle="none",
              markers="D", errorbar=("ci", 95), capsize=0.2, ax=axes[2])
axes[2].set_title("swarm + średnia z 95% CI (pointplot)")
plt.show()

### 7.5. Pointplot: `sns.pointplot`

**Kiedy:** porównanie **średnich (lub innego estymatora) z przedziałem ufności** między kategoriami, zwłaszcza przy 2 czynnikach (interakcja). Linie łączą punkty jednego `hue`, więc **równoległe linie = brak interakcji**.

Parametry: `x`, `y` ✅; `hue`, `order`, `estimator`, `errorbar`, `capsize`, `dodge` (np. `0.3` rozsuwa serie), `linestyle` (`"none"` bez linii), `markers`, `linestyles`, `markersize`, `linewidth`, `palette`, `native_scale`.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.pointplot(df, x="segment", y="satisfaction", hue="channel", dodge=0.3,
              errorbar=("ci", 95), capsize=0.1, markers=["o", "s", "^"], linestyles=["-", "--", ":"],
              palette="Set2", ax=ax)
ax.set_title("Satysfakcja: segment × kanał (średnia + 95% CI)")
plt.show()

### 7.6. Countplot: `sns.countplot`

**Kiedy:** liczności kategorii. To `barplot` z `estimator="count"`. **Interpretacja:** wysokość = liczba obserwacji (lub % przy `stat`). Sortuj kategorie nominalne malejąco; kategorie porządkowe zostaw w naturalnej kolejności.

Parametry: `data`, `x` (lub `y`) ✅; `hue`, `order`, `hue_order`, `stat` (`"count"`, `"percent"`, `"proportion"`, `"probability"`), `dodge`, `palette`, `saturation`, `width`, `ax`.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), constrained_layout=True)

order = df["region"].value_counts().index
sns.countplot(df, x="region", order=order, hue="region", palette="viridis", legend=False, ax=axes[0])
axes[0].bar_label(axes[0].containers[0], padding=2)
axes[0].set_title("Posortowane, z etykietami")

sns.countplot(df, y="channel", hue="segment", stat="percent", palette="rocket", ax=axes[1])
axes[1].set_title("Poziomy, hue, stat='percent'")
axes[1].xaxis.set_major_formatter(mticker.PercentFormatter(100))

sns.countplot(df, x="satisfaction", hue="is_returning", palette={True: "#2a9d8f", False: "#e9c46a"}, ax=axes[2])
axes[2].set_title("Zmienna porządkowa × bool")
plt.show()

### 7.7. `sns.catplot` (figure-level dla wykresów kategorycznych)

`kind`: `"strip"`, `"swarm"`, `"box"`, `"violin"`, `"boxen"`, `"point"`, `"bar"`, `"count"`. Dodaje `col=`, `row=`, `col_wrap=`, `height=`, `aspect=`, `sharey=`, `margin_titles=`.

In [ ]:
g = sns.catplot(df, x="segment", y="revenue", hue="is_returning", col="channel", kind="box",
                palette={True: "#2a9d8f", False: "#e76f51"}, height=4, aspect=0.9,
                showfliers=False, sharey=True)
g.set_axis_labels("Segment", "Przychód")
g.set_titles("Kanał: {col_name}")
g.add_legend(title="Powracający")
plt.show()

## 8. Zależności między dwiema zmiennymi ciągłymi

### 8.1. Scatterplot: `sns.scatterplot`

**Kiedy:** związek dwóch zmiennych ciągłych; do 4–5 wymiarów jednocześnie (x, y, kolor, rozmiar, kształt).
**Interpretacja:** kierunek (dodatni/ujemny), siła (jak ciasna chmura), kształt (liniowy/nieliniowy), klastry, outliery, heteroskedastyczność (lejek). **Korelacja ≠ przyczynowość**, a współczynnik r mierzy tylko zależność liniową.
**Overplotting (> ~5 000 punktów):** zmniejsz `s`, dodaj `alpha`, użyj hexbin / histplot 2D / KDE.

| Parametr | Opis |
|---|---|
| `data`, `x`, `y` ✅ | |
| `hue` ⬜ | Kolor: kategoria lub zmienna ciągła (`palette="viridis"`, `hue_norm`) |
| `size` ⬜ | Rozmiar wg zmiennej; `sizes=(min, max)`, `size_norm` |
| `style` ⬜ | Kształt znacznika wg kategorii; `markers` dict/lista |
| `s` ⬜ | Stały rozmiar (gdy bez `size`) |
| `alpha` ⬜ | Przezroczystość |
| `edgecolor`, `linewidth` ⬜ | Obrys punktów |
| `marker` ⬜ | Znacznik stały |
| `legend` ⬜ | `"auto"`, `"brief"`, `"full"`, `False` |
| `palette`, `hue_order` ⬜ | |

In [ ]:
smp = df.sample(600, random_state=4)
fig, axes = plt.subplots(2, 2, figsize=(14, 11), constrained_layout=True)

# a) hue kategoryczne + style
sns.scatterplot(smp, x="unit_price", y="revenue", hue="segment", style="channel",
                palette="viridis", s=45, alpha=0.7, edgecolor="white", linewidth=0.4, ax=axes[0, 0])
axes[0, 0].set_title("hue + style")

# b) hue ciągłe + size
sns.scatterplot(smp, x="unit_price", y="revenue", hue="discount", size="units",
                palette="flare", sizes=(15, 200), alpha=0.7, ax=axes[0, 1])
axes[0, 1].legend(loc="upper left", fontsize=8, ncol=2)
axes[0, 1].set_title("hue (ciągłe) + size")

# c) wyróżnienie podzbioru + adnotacje + linie
sns.scatterplot(smp, x="unit_price", y="revenue", color="#c7cdd6", s=25, ax=axes[1, 0])
top = smp.nlargest(5, "revenue")
axes[1, 0].scatter(top["unit_price"], top["revenue"], s=110, color="#d62728", edgecolor="black", zorder=3, label="Top 5")
for _, r in top.iterrows():
    axes[1, 0].annotate(r["order_id"], (r["unit_price"], r["revenue"]), xytext=(6, 6),
                        textcoords="offset points", fontsize=8)
axes[1, 0].axhline(smp["revenue"].quantile(0.95), color="black", linestyle="--", linewidth=1)
axes[1, 0].text(smp["unit_price"].min(), smp["revenue"].quantile(0.95), " P95", va="bottom", fontsize=9)
axes[1, 0].legend()
axes[1, 0].set_title("Wyróżnione punkty, adnotacje, próg P95")

# d) ręczna linia regresji (polyfit) + r + pasmo
x, y = smp["unit_price"].to_numpy(), smp["revenue"].to_numpy()
slope, intercept = np.polyfit(x, y, 1)
r = np.corrcoef(x, y)[0, 1]
xs = np.linspace(x.min(), x.max(), 100)
resid_sd = np.std(y - (slope * x + intercept))
sns.scatterplot(smp, x="unit_price", y="revenue", color="#4c72b0", alpha=0.5, ax=axes[1, 1])
axes[1, 1].plot(xs, slope * xs + intercept, color="crimson", linewidth=2, label=f"y = {slope:.2f}x + {intercept:.1f}")
axes[1, 1].fill_between(xs, slope * xs + intercept - 2 * resid_sd, slope * xs + intercept + 2 * resid_sd,
                        color="crimson", alpha=0.1, label="±2 σ reszt")
axes[1, 1].text(0.05, 0.92, f"r = {r:.2f}", transform=axes[1, 1].transAxes, fontsize=12,
                bbox=dict(boxstyle="round", facecolor="white", alpha=0.8))
axes[1, 1].legend(loc="lower right")
axes[1, 1].set_title("Regresja liniowa ręcznie (np.polyfit)")
plt.show()

### 8.2. Relplot: `sns.relplot` (figure-level scatter/line z panelami)

`kind="scatter"` (domyślnie) lub `"line"`; wszystkie parametry `scatterplot`/`lineplot` + `col`, `row`, `col_wrap`, `height`, `aspect`, `facet_kws`.

In [ ]:
g = sns.relplot(smp, x="unit_price", y="revenue", hue="segment", col="channel", row="is_returning",
                kind="scatter", palette="viridis", height=3, aspect=1.2, alpha=0.7, s=30,
                facet_kws={"sharex": True, "sharey": True, "margin_titles": True})
g.set_titles(col_template="{col_name}", row_template="powracający: {row_name}")
g.refline(y=smp["revenue"].median(), color="gray", linestyle="--")      # linia referencyjna w każdym panelu
plt.show()

g = sns.relplot(df_month, x="month", y="revenue", hue="channel", col="channel", kind="line",
                col_wrap=3, height=3, aspect=1.4, marker="o", legend=False, palette="Set2")
g.set_xticklabels(rotation=45)
plt.show()

### 8.3. `sns.regplot` i `sns.lmplot`: regresja

**Kiedy:** zobaczyć trend i jego niepewność. `regplot` = axes-level, `lmplot` = figure-level (`hue`, `col`, `row`).
**Interpretacja:** linia = dopasowanie modelu; pasmo = 95% CI **średniej** predykcji (nie przedział predykcji dla pojedynczej obserwacji!). Różne nachylenia linii dla `hue` = interakcja.

| Parametr | Opis |
|---|---|
| `data`, `x`, `y` ✅ | |
| `order` ⬜ | Stopień wielomianu (`2` = parabola) |
| `robust` ⬜ | Regresja odporna na outliery (statsmodels) |
| `logistic` ⬜ | Regresja logistyczna (`y` binarne 0/1) |
| `lowess` ⬜ | Lokalna regresja nieparametryczna (nieliniowy trend bez modelu) |
| `ci` ⬜ | Poziom ufności (95), `None` wyłącza pasmo |
| `truncate` ⬜ | `True` = linia tylko w zakresie danych |
| `x_estimator`, `x_bins` ⬜ | Binning X i agregacja (czytelne przy dużych danych) |
| `x_jitter`, `y_jitter` ⬜ | Jitter dla dyskretnych zmiennych |
| `scatter_kws`, `line_kws` ⬜ | Style punktów / linii (`{"s": 20, "alpha": .4}`, `{"color": "red"}`) |
| `scatter`, `fit_reg` ⬜ | Wyłączanie punktów / linii |
| `hue`, `col`, `row`, `height`, `aspect`, `palette`, `markers`, `sharex`, `sharey` ⬜ | (lmplot) |

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(17, 9), constrained_layout=True)
sk = dict(s=18, alpha=0.35)

sns.regplot(smp, x="customer_age", y="income", scatter_kws=sk, line_kws={"color": "crimson"}, ax=axes[0, 0])
axes[0, 0].set_title("Liniowa + 95% CI")

sns.regplot(smp, x="delivery_days", y="satisfaction", order=2, x_jitter=0, y_jitter=0.15,
            scatter_kws=sk, line_kws={"color": "darkgreen"}, ax=axes[0, 1])
axes[0, 1].set_title("order=2 + y_jitter")

sns.regplot(smp, x="customer_age", y="income", lowess=True, scatter_kws=sk, line_kws={"color": "purple"}, ax=axes[0, 2])
axes[0, 2].set_title("lowess (nieparametryczna)")

sns.regplot(smp, x="delivery_days", y="satisfaction", x_bins=8, fit_reg=True, ci=95,
            scatter_kws={"s": 60}, ax=axes[1, 0])
axes[1, 0].set_title("x_bins=8 (średnie w przedziałach)")

sns.regplot(smp, x="income", y="returning_int", logistic=True, y_jitter=0.03,
            scatter_kws={"s": 12, "alpha": 0.4}, ax=axes[1, 1])
axes[1, 1].set_title("Regresja logistyczna")

sns.regplot(smp, x="customer_age", y="income", robust=True, ci=None, scatter_kws=sk, ax=axes[1, 2])
axes[1, 2].set_title("robust=True, bez pasma")
plt.show()

# lmplot: osobna regresja per grupa + panele
g = sns.lmplot(smp, x="unit_price", y="revenue", hue="segment", col="channel", palette="viridis",
               height=4, aspect=1, scatter_kws={"s": 20, "alpha": 0.5}, line_kws={"linewidth": 2},
               truncate=True, ci=90, facet_kws={"sharey": False})
g.set_titles("Kanał: {col_name}")
plt.show()

### 8.4. Residplot: `sns.residplot`

**Kiedy:** diagnostyka regresji liniowej. **Interpretacja:** reszty powinny tworzyć losową chmurę wokół 0. Wzorzec (U, lejek) = zła forma funkcyjna lub heteroskedastyczność. Parametry: `x`, `y` ✅, `order`, `lowess` (pokazuje trend reszt), `robust`, `scatter_kws`, `line_kws`.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5), constrained_layout=True)
sns.residplot(smp, x="unit_price", y="revenue", lowess=True, scatter_kws={"alpha": 0.4, "s": 20},
              line_kws={"color": "crimson"}, ax=axes[0])
axes[0].set_title("Reszty: lejek ⇒ heteroskedastyczność")
sns.residplot(smp, x="unit_price", y="revenue", order=2, scatter_kws={"alpha": 0.4, "s": 20}, ax=axes[1])
axes[1].set_title("Reszty modelu kwadratowego")
plt.show()

## 9. Wykresy dwuwymiarowe i macierzowe układy

### 9.1. Jointplot: `sns.jointplot`

**Kiedy:** zależność X–Y **razem z rozkładami brzegowymi**. Figure-level (nie przyjmuje `ax`).

| Parametr | Opis |
|---|---|
| `data`, `x`, `y` ✅ | |
| `kind` ⬜ | `"scatter"`, `"kde"`, `"hist"`, `"hex"`, `"reg"`, `"resid"` |
| `hue` ⬜ | Grupy (nie działa z `hex`, `reg`, `resid`) |
| `height`, `ratio`, `space` ⬜ | Rozmiar, proporcja panelu głównego do brzegowych, odstęp |
| `marginal_ticks` ⬜ | Ticki na panelach brzegowych |
| `joint_kws`, `marginal_kws` ⬜ | Parametry dla panelu głównego i brzegowych |
| `xlim`, `ylim` ⬜ | Zakresy |
| `palette`, `color`, `dropna` ⬜ | |

In [ ]:
g = sns.jointplot(df, x="unit_price", y="revenue", hue="segment", kind="scatter", palette="viridis",
                  height=6, ratio=4, space=0.1, alpha=0.5, s=20,
                  marginal_kws={"fill": True, "common_norm": False})
g.figure.suptitle("jointplot: scatter + KDE brzegowe", y=1.02)
plt.show()

g = sns.jointplot(df, x="unit_price", y="revenue", kind="hex", height=6, color="#264653",
                  joint_kws={"gridsize": 30, "mincnt": 1, "cmap": "mako_r"},
                  marginal_kws={"bins": 30})
plt.show()

g = sns.jointplot(df, x="unit_price", y="revenue", kind="kde", fill=True, cmap="rocket_r", thresh=0.05, levels=8, height=6)
plt.show()

g = sns.jointplot(smp, x="customer_age", y="income", kind="reg", height=6,
                  joint_kws={"line_kws": {"color": "crimson"}, "scatter_kws": {"alpha": 0.4}})
r = smp["customer_age"].corr(smp["income"])
g.ax_joint.text(0.05, 0.92, f"r = {r:.2f}", transform=g.ax_joint.transAxes)
plt.show()

**Pełna kontrola nad własnym układem: `JointGrid`:**

In [ ]:
g = sns.JointGrid(data=df, x="unit_price", y="revenue", hue="channel", height=6, ratio=5, space=0.05)
g.plot_joint(sns.scatterplot, s=15, alpha=0.5)
g.plot_marginals(sns.histplot, element="step", fill=False, kde=True)
g.refline(x=df["unit_price"].median(), y=df["revenue"].median(), color="gray", linestyle="--")
plt.show()

### 9.2. Hexbin (matplotlib) i histogram 2D

**Kiedy:** bardzo dużo punktów (tysiące–miliony), gdzie scatter zamienia się w plamę. **Interpretacja:** kolor sześciokąta = liczba punktów (lub agregat zmiennej `C`).

| Parametr `ax.hexbin` | Opis |
|---|---|
| `x`, `y` ✅ | |
| `C` ⬜ | Zmienna do agregacji w komórce (zamiast liczności) |
| `reduce_C_function` ⬜ | Funkcja agregująca `C` (`np.mean`, `np.median`, `np.sum`) |
| `gridsize` ⬜ | Liczba komórek (większa = drobniej) |
| `bins` ⬜ | `None` (liniowo), `"log"` (kolory w skali log) |
| `mincnt` ⬜ | Minimalna liczba punktów do narysowania komórki |
| `cmap`, `extent`, `norm`, `vmin`, `vmax`, `linewidths`, `edgecolors` ⬜ | |

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(17, 4.8), constrained_layout=True)

hb = axes[0].hexbin(df["unit_price"], df["revenue"], gridsize=30, cmap="mako_r", mincnt=1)
fig.colorbar(hb, ax=axes[0], label="Liczba zamówień")
axes[0].set_title("hexbin: liczność")

hb = axes[1].hexbin(df["unit_price"], df["revenue"], C=df["discount"], reduce_C_function=np.mean,
                    gridsize=25, cmap="rocket_r", mincnt=3)
fig.colorbar(hb, ax=axes[1], label="Średni rabat")
axes[1].set_title("hexbin: średnia trzeciej zmiennej (C)")

hb = axes[2].hexbin(df["unit_price"], df["revenue"], gridsize=30, bins="log", cmap="viridis")
fig.colorbar(hb, ax=axes[2], label="log10(liczność)")
axes[2].set_title("hexbin: bins='log'")
for ax in axes:
    ax.set_xlabel("unit_price"); ax.set_ylabel("revenue")
plt.show()

### 9.3. Pairplot: `sns.pairplot`

**Kiedy:** pierwszy przegląd (EDA) wielu zmiennych liczbowych naraz: rozkłady (przekątna) + relacje par (poza przekątną).

| Parametr | Opis |
|---|---|
| `data` ✅ | DataFrame |
| `vars`, `x_vars`, `y_vars` ⬜ | Wybór kolumn (domyślnie wszystkie liczbowe; **ogranicz do ~5–6**) |
| `hue` ⬜ | Grupy |
| `kind` ⬜ | `"scatter"`, `"kde"`, `"hist"`, `"reg"` |
| `diag_kind` ⬜ | `"auto"`, `"hist"`, `"kde"`, `None` |
| `corner` ⬜ | `True` = tylko dolny trójkąt (bez duplikatów) |
| `plot_kws`, `diag_kws` ⬜ | Parametry wykresów |
| `height`, `aspect`, `palette`, `markers` ⬜ | |

In [ ]:
g = sns.pairplot(smp, vars=["unit_price", "revenue", "delivery_days", "satisfaction"], hue="segment",
                 corner=True, diag_kind="kde", palette="viridis", height=2.2,
                 plot_kws={"alpha": 0.5, "s": 15}, diag_kws={"fill": True, "common_norm": False})
plt.show()

# pełna kontrola: PairGrid
g = sns.PairGrid(smp, vars=["unit_price", "revenue", "delivery_days"], hue="channel", palette="Set2", height=2.6)
g.map_upper(sns.scatterplot, s=15, alpha=0.5)
g.map_lower(sns.kdeplot, levels=4)
g.map_diag(sns.histplot, element="step", fill=False)
g.add_legend()
plt.show()

## 10. Macierze: heatmap i clustermap

### 10.1. Heatmap: `sns.heatmap`

**Kiedy:** macierz korelacji, tabela przestawna (kategoria × kategoria), macierz pomyłek, kalendarz aktywności.
**Interpretacja:** kolor = wartość komórki. Dla korelacji używaj palety **rozbieżnej** z `center=0` i `vmin=-1, vmax=1`; dla wartości nieujemnych sekwencyjnej. Dodaj `annot=True` przy małych macierzach (< ~15×15).

| Parametr | Opis |
|---|---|
| `data` ✅ | Macierz 2D (DataFrame, najlepiej z etykietami) |
| `vmin`, `vmax` ⬜ | Zakres kolorów |
| `center` ⬜ | Wartość środkowa palety rozbieżnej (np. `0`) |
| `cmap` ⬜ | Colormapa (`"coolwarm"`, `"vlag"`, `"rocket"`, `"YlGnBu"`) |
| `annot` ⬜ | `True` lub macierz własnych etykiet |
| `fmt` ⬜ | Format liczb: `".2f"`, `",.0f"`, `".0%"`, `"d"` |
| `annot_kws` ⬜ | `{"size": 9, "weight": "bold"}` |
| `linewidths`, `linecolor` ⬜ | Siatka między komórkami |
| `cbar`, `cbar_kws` ⬜ | Pasek kolorów: `{"label": "r", "shrink": .8, "orientation": "horizontal"}` |
| `square` ⬜ | Kwadratowe komórki |
| `mask` ⬜ | Macierz bool, `True` = komórka ukryta (np. górny trójkąt) |
| `xticklabels`, `yticklabels` ⬜ | `"auto"`, `True`, `False`, krok (int) |
| `robust` ⬜ | Zakres kolorów z percentyli 2–98 (odporne na outliery) |

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(15, 12), constrained_layout=True)

# a) korelacje: dolny trójkąt, rozbieżna paleta wycentrowana na 0
mask = np.triu(np.ones_like(corr, dtype=bool))
sns.heatmap(corr, mask=mask, annot=True, fmt=".2f", cmap="vlag", center=0, vmin=-1, vmax=1,
            square=True, linewidths=0.5, linecolor="white", cbar_kws={"shrink": 0.7, "label": "Pearson r"},
            annot_kws={"size": 9}, ax=axes[0, 0])
axes[0, 0].set_title("Macierz korelacji (dolny trójkąt)")

# b) tabela przestawna, sekwencyjna paleta
sns.heatmap(pivot, annot=True, fmt=",.0f", cmap="YlGnBu", linewidths=0.5, cbar_kws={"label": "Śr. przychód"}, ax=axes[0, 1])
axes[0, 1].set_title("Region × segment")

# c) kalendarz aktywności: liczba zamówień wg miesiąca i dnia tygodnia
days = ["Monday", "Tuesday", "Wednesday", "Thursday", "Friday", "Saturday", "Sunday"]
cal = (df.assign(m=df["order_date"].dt.month)
         .pivot_table(index="weekday", columns="m", values="order_id", aggfunc="count")
         .reindex(days))
sns.heatmap(cal, cmap="rocket_r", annot=True, fmt=".0f", linewidths=1, linecolor="white", ax=axes[1, 0])
axes[1, 0].set_title("Zamówienia: dzień tygodnia × miesiąc")
axes[1, 0].set_xlabel("Miesiąc"); axes[1, 0].set_ylabel("")

# d) obramowanie wybranej komórki i etykiety własne (annot jako macierz)
share = pivot.div(pivot.sum(axis=1), axis=0)
labels = pivot.round(0).astype(int).astype(str) + "\n(" + (share * 100).round(0).astype(int).astype(str) + "%)"
sns.heatmap(share, annot=labels, fmt="", cmap="crest", cbar=False, ax=axes[1, 1])
r_i, c_i = np.unravel_index(np.argmax(share.to_numpy()), share.shape)       # komórka z maksimum
axes[1, 1].add_patch(Rectangle((c_i, r_i), 1, 1, fill=False, edgecolor="red", linewidth=3))
axes[1, 1].set_title("Etykiety własne + obramowana komórka max")
plt.show()

### 10.2. Clustermap: `sns.clustermap`

**Kiedy:** grupowanie hierarchiczne wierszy/kolumn. Odkrywanie grup skorelowanych zmiennych, segmentów klientów, podobnych obiektów. Figure-level.
**Interpretacja:** wiersze/kolumny są **przestawione** tak, by podobne leżały obok siebie; dendrogram pokazuje kolejność łączenia (im niżej połączenie, tym większe podobieństwo). Bloki o podobnym kolorze = klastry.

| Parametr | Opis |
|---|---|
| `data` ✅ | Macierz |
| `method` ⬜ | Metoda łączenia: `"average"`, `"ward"`, `"complete"`, `"single"` (`"ward"` wymaga metryki euklidesowej) |
| `metric` ⬜ | Odległość: `"euclidean"`, `"correlation"`, `"cosine"`… |
| `standard_scale` / `z_score` ⬜ | Skalowanie `0` (wiersze) lub `1` (kolumny); **konieczne, gdy kolumny mają różne jednostki** |
| `row_cluster`, `col_cluster` ⬜ | Wyłączenie klastrowania danej osi |
| `row_colors`, `col_colors` ⬜ | Paski z kategorią (np. segment) obok wierszy/kolumn |
| `figsize`, `dendrogram_ratio`, `cbar_pos`, `colors_ratio` ⬜ | Układ |
| `cmap`, `center`, `vmin`, `vmax`, `annot`, `fmt`, `linewidths` ⬜ | Jak w `heatmap` |

In [ ]:
# a) klastrowanie zmiennych po korelacji
g = sns.clustermap(corr, method="average", metric="euclidean", cmap="vlag", center=0, vmin=-1, vmax=1,
                   annot=True, fmt=".2f", linewidths=0.5, figsize=(8, 8), dendrogram_ratio=(0.15, 0.15),
                   cbar_pos=(0.02, 0.8, 0.03, 0.15))
plt.show()

# b) klastrowanie obserwacji (wiersze) ze skalowaniem kolumn + pasek segmentu
num_cols = ["customer_age", "income", "units", "unit_price", "discount", "revenue", "delivery_days", "satisfaction"]
sub = df.dropna(subset=num_cols).sample(80, random_state=5)
lut = dict(zip(df["segment"].cat.categories, sns.color_palette("Set2", 3)))
row_colors = sub["segment"].astype(str).map(lambda s: lut[s])
row_colors.name = "segment"

g = sns.clustermap(sub[num_cols], z_score=1, method="ward", cmap="vlag", center=0, robust=True,
                   row_colors=row_colors.to_numpy(), yticklabels=False, figsize=(9, 8),
                   dendrogram_ratio=(0.12, 0.15), cbar_kws={"label": "z-score"})
g.ax_heatmap.set_xlabel("")
plt.show()

## 11. Ridge plot (joyplot)

**Kiedy:** porównanie **rozkładów wielu grup** (≥ 4–5), gdy KDE nałożone na siebie byłyby nieczytelne, a violin zajmuje zbyt dużo miejsca. Klasyczne zastosowanie: rozkład w czasie (miesiące, lata).
**Interpretacja:** każda „fala" = gęstość jednej grupy; przesunięcie fali w prawo = wyższe wartości; szerokość = rozrzut. Wspólna oś X umożliwia bezpośrednie porównanie. Brak własnej funkcji w seaborn: budujemy z `FacetGrid` (alternatywa: biblioteka `joypy`).

In [ ]:
order = df.groupby("region")["income"].median().sort_values().index.tolist()
pal = sns.cubehelix_palette(len(order), rot=-0.25, light=0.7)

with sns.axes_style("white", rc={"axes.facecolor": (0, 0, 0, 0)}):            # przezroczyste tło osi (nakładanie)
    g = sns.FacetGrid(df, row="region", hue="region", row_order=order, hue_order=order,
                      aspect=8, height=0.9, palette=pal)

g.map_dataframe(sns.kdeplot, x="income", fill=True, alpha=0.85, linewidth=0, bw_adjust=0.9, clip_on=False)
g.map_dataframe(sns.kdeplot, x="income", color="white", linewidth=2, bw_adjust=0.9, clip_on=False)
g.refline(y=0, linewidth=1.5, linestyle="-", color=None, clip_on=False)       # linia bazowa każdej fali

for ax, name in zip(g.axes.flat, g.row_names):                                # etykiety grup na osi
    ax.text(0.0, 0.15, name, fontweight="bold", transform=ax.transAxes)

for ax, name in zip(g.axes.flat, order):                                      # mediany jako pionowe kreski
    med = df.loc[df["region"] == name, "income"].median()
    ax.plot([med, med], [0, ax.get_ylim()[1] * 0.5], color="black", linewidth=1.5, clip_on=False)

g.figure.subplots_adjust(hspace=-0.55)                                        # ujemny odstęp = nakładanie fal
g.set_titles("")
g.set(yticks=[], ylabel="")
g.despine(bottom=True, left=True)
g.set_xlabels("Dochód klienta")
g.figure.suptitle("Rozkład dochodu wg regionu (czarna kreska = mediana)", y=0.98, fontweight="bold")
plt.show()

**Wersja czysto matplotlib** (pełna kontrola nad wysokością i przesunięciem fal):

In [ ]:
from scipy.stats import gaussian_kde

fig, ax = plt.subplots(figsize=(8, 5))
grid = np.linspace(df["income"].min(), df["income"].max(), 300)
for i, name in enumerate(order):
    vals = df.loc[df["region"] == name, "income"].dropna()
    dens = gaussian_kde(vals)(grid)
    dens = dens / dens.max() * 1.8                                    # skalowanie wysokości fali
    ax.fill_between(grid, i, i + dens, color=pal[i], alpha=0.9, zorder=len(order) - i)
    ax.plot(grid, i + dens, color="white", linewidth=1.5, zorder=len(order) - i)
ax.set_yticks(range(len(order)))
ax.set_yticklabels(order)
ax.set_xlabel("Dochód klienta")
ax.spines[["left", "top", "right"]].set_visible(False)
ax.grid(False)
plt.show()

## 12. Układy złożone: FacetGrid, dual axis, dashboard

### 12.1. `FacetGrid`: dowolny wykres w panelach

**Kiedy:** ten sam wykres per kategoria (małe wielokrotności, *small multiples*). Najskuteczniejsza technika porównań wielu grup.

| Parametr `FacetGrid` | Opis |
|---|---|
| `data` ✅ | DataFrame |
| `row`, `col`, `hue` ⬜ | Zmienne dzielące na panele / kolory |
| `col_wrap` ⬜ | Zawijanie paneli (tylko z `col`) |
| `height`, `aspect` ⬜ | Rozmiar panelu (wys., szer./wys.) |
| `sharex`, `sharey` ⬜ | Wspólne skale |
| `row_order`, `col_order`, `hue_order`, `palette` ⬜ | |
| `margin_titles` ⬜ | Tytuły wierszy na marginesie |

Metody: `g.map_dataframe(func, **kws)` (w `func` podajesz nazwy kolumn jako `x="..."`, `y="..."`), `g.map(func, "kolumna")`, `g.set_titles()`, `g.set_axis_labels()`, `g.refline()`, `g.add_legend()`, `g.tight_layout()`.

In [ ]:
g = sns.FacetGrid(df, col="region", col_wrap=3, height=3.2, aspect=1.2, sharex=True, sharey=True, palette="Set2")
g.map_dataframe(sns.histplot, x="revenue", bins=25, color="#4c72b0")
g.map_dataframe(sns.rugplot, x="revenue", color="black", alpha=0.1)
g.refline(x=df["revenue"].median(), color="red", linestyle="--")              # globalna mediana w każdym panelu
g.set_titles("{col_name}")
g.set_axis_labels("Przychód", "Liczba")
plt.show()

### 12.2. Dual axis (druga oś Y)

**Kiedy:** dwie miary w różnych jednostkach na wspólnej osi czasu (np. przychód + liczba zamówień). **Uwaga:** łatwo o mylący efekt (skale dobrane tak, by linie „pasowały"). Używaj oszczędnie i zawsze koloruj osie zgodnie z seriami.

In [ ]:
tot = df.groupby("month").agg(revenue=("revenue", "sum"), orders=("order_id", "size"))

fig, ax1 = plt.subplots(figsize=(10, 4.5), constrained_layout=True)
ax1.bar(tot.index, tot["revenue"], width=20, color="#8da0cb", label="Przychód")
ax1.set_ylabel("Przychód [PLN]", color="#4c72b0")
ax1.tick_params(axis="y", labelcolor="#4c72b0")
ax1.yaxis.set_major_formatter(mticker.FuncFormatter(lambda v, _: f"{v/1e3:,.0f}k"))

ax2 = ax1.twinx()
ax2.plot(tot.index, tot["orders"], color="#d62728", marker="o", linewidth=2, label="Zamówienia")
ax2.set_ylabel("Liczba zamówień", color="#d62728")
ax2.tick_params(axis="y", labelcolor="#d62728")
ax2.grid(False)
ax2.spines["right"].set_visible(True)

h1, l1 = ax1.get_legend_handles_labels()                                        # wspólna legenda
h2, l2 = ax2.get_legend_handles_labels()
ax1.legend(h1 + h2, l1 + l2, loc="upper left")
ax1.xaxis.set_major_formatter(mdates.DateFormatter("%b %y"))
plt.show()

### 12.3. Mini-dashboard (mozaika + seaborn)

In [ ]:
fig, axd = plt.subplot_mosaic(
    [["trend", "trend", "pie"],
     ["box", "heat", "heat"]],
    figsize=(15, 8), constrained_layout=True, gridspec_kw={"height_ratios": [1.2, 1]})

sns.lineplot(df_month, x="month", y="revenue", hue="channel", palette="Set2", ax=axd["trend"])
axd["trend"].set_title("Trend przychodu wg kanału")

vals = df.groupby("channel")["revenue"].sum()
axd["pie"].pie(vals, labels=vals.index, autopct="%1.0f%%", startangle=90,
               colors=sns.color_palette("Set2", 3), wedgeprops={"width": 0.45, "edgecolor": "white"})
axd["pie"].set_title("Udział kanałów")

sns.boxplot(df, x="segment", y="revenue", hue="segment", palette="viridis", legend=False, showfliers=False, ax=axd["box"])
axd["box"].set_title("Przychód wg segmentu")

sns.heatmap(pivot, annot=True, fmt=",.0f", cmap="YlGnBu", cbar=False, ax=axd["heat"])
axd["heat"].set_title("Śr. przychód: region × segment")

fig.suptitle("Dashboard sprzedaży", fontsize=16, fontweight="bold")
plt.show()

## 13. Zbiorczo: linie, cieniowanie i wyróżnianie elementów

### 13.1. Linie

| Cel | Matplotlib (`ax.`) | Seaborn / uwagi |
|---|---|---|
| Pozioma / pionowa linia | `axhline(y, xmin, xmax)`, `axvline(x, ymin, ymax)` | `g.refline(x=, y=)` dla grid-ów |
| Linia o zadanym nachyleniu | `axline((x0, y0), slope=1)` | Przekątna „y = x" (np. predykcja vs rzeczywistość) |
| Odcinki z danych | `hlines(y, xmin, xmax)`, `vlines(x, ymin, ymax)` | Lollipopy, zakresy |
| Dowolna linia | `ax.plot([x0, x1], [y0, y1])` | |
| Styl | `color`, `linestyle` (`"-"`, `"--"`, `":"`, `"-."`), `linewidth`, `alpha`, `zorder`, `label` | |

### 13.2. Cieniowanie

| Cel | Metoda |
|---|---|
| Pas poziomy / pionowy | `axhspan(ymin, ymax)`, `axvspan(xmin, xmax)` (`alpha=0.1–0.3`) |
| Pas wokół krzywej | `fill_between(x, y1, y2, where=, interpolate=True)` |
| Pas w poziomie (X) | `fill_betweenx(y, x1, x2)` |
| Wypełnienie warunkowe | `where=(y > próg)` w `fill_between` |
| Prostokąt dowolny | `ax.add_patch(Rectangle((x, y), w, h, alpha=0.2))` |

### 13.3. Wyróżnianie wybranych punktów, słupków, kategorii

| Co wyróżniamy | Sposób |
|---|---|
| Słupki w `barplot` | Słownik `palette={kategoria: kolor}` + `hue=` (zalecane), lub pętla po `ax.patches` (`set_facecolor`, `set_hatch`, `set_edgecolor`) |
| Punkty w `scatterplot` | Rysuj całość szarym kolorem, a podzbiór drugim `scatter`/`scatterplot` z większym `s` i `zorder=3` |
| Punkty warunkowo | `mask = df["x"] > próg`; `ax.scatter(df.loc[mask, "x"], df.loc[mask, "y"], color="red")` |
| Linia / seria | Szare tło dla wszystkich serii (`alpha=.3`) + pogrubiona wybrana seria |
| Etykiety słupków | `ax.bar_label(ax.containers[i], fmt=, padding=, label_type="edge"/"center")` |
| Etykiety ticków | `for lbl in ax.get_xticklabels(): if lbl.get_text() == "Centrum": lbl.set_color("red"); lbl.set_fontweight("bold")` |

In [ ]:
# przykład: seria wyróżniona na tle pozostałych
pv = df.pivot_table(index="month", columns="region", values="revenue", aggfunc="sum")
fig, ax = plt.subplots(figsize=(9, 4.5), constrained_layout=True)
for col in pv.columns:
    ax.plot(pv.index, pv[col], color="lightgray", linewidth=1.5)
ax.plot(pv.index, pv["Centrum"], color="#d62728", linewidth=3, label="Centrum")
ax.text(pv.index[-1], pv["Centrum"].iloc[-1], "  Centrum", color="#d62728", va="center", fontweight="bold")
ax.set_title("Wyróżniony region na tle pozostałych")
plt.show()

# przykład: wyróżnienie etykiety tickowej
fig, ax = plt.subplots(figsize=(7, 4))
tot_r = df.groupby("region")["revenue"].sum().sort_values(ascending=False)
ax.bar(tot_r.index, tot_r.values, color="#8da0cb")
for lbl in ax.get_xticklabels():
    if lbl.get_text() == "Centrum":
        lbl.set_color("red"); lbl.set_fontweight("bold")
plt.show()

### 13.4. Adnotacje: parametry `ax.annotate`

| Parametr | Opis |
|---|---|
| `text` ✅, `xy` ✅ | Treść i punkt, na który wskazujemy |
| `xytext` ⬜ | Pozycja tekstu |
| `textcoords` ⬜ | `"data"`, `"offset points"`, `"axes fraction"` |
| `arrowprops` ⬜ | `dict(arrowstyle="->", color=, lw=, connectionstyle="arc3,rad=0.2")` |
| `ha`, `va`, `fontsize`, `color`, `fontweight` ⬜ | |
| `bbox` ⬜ | `dict(boxstyle="round,pad=0.3", fc="white", ec="gray", alpha=0.9)` |

## 14. Zapis, wydajność, pułapki

### 14.1. Zapis do pliku

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
sns.barplot(df, x="region", y="revenue", hue="region", legend=False, ax=ax)
fig.savefig("wykres.png", dpi=300, bbox_inches="tight", facecolor="white", transparent=False)  # PNG do raportów
fig.savefig("wykres.svg", bbox_inches="tight")                                                  # wektor (skalowalny)
fig.savefig("wykres.pdf", bbox_inches="tight")                                                  # PDF do druku
plt.show()

# figure-level: g.savefig("plik.png", dpi=300, bbox_inches="tight")

| Parametr `savefig` | Opis |
|---|---|
| `fname` ✅ | Ścieżka; format z rozszerzenia (`png`, `svg`, `pdf`, `jpg`) |
| `dpi` ⬜ | Rozdzielczość (150 ekran, 300 druk) |
| `bbox_inches` ⬜ | `"tight"` przycina marginesy |
| `facecolor`, `transparent` ⬜ | Tło / przezroczyste tło |
| `pad_inches` ⬜ | Margines przy `tight` |

W Power BI wykresy z Pythona (visual Python) dostają dane jako `dataset` (pandas), a rysujesz tak samo jak powyżej. Nie wywołuj `savefig`, wystarczy `plt.show()`; pamiętaj, że limit to ok. 150 000 wierszy.

### 14.2. Wydajność przy dużych danych

| Problem | Rozwiązanie |
|---|---|
| Scatter z 100k+ punktów jest wolny i nieczytelny | `hexbin` / `histplot(x, y)` / `kdeplot` 2D lub próbkowanie `df.sample(5000)`; dla milionów punktów biblioteka `datashader` |
| `lineplot`/`barplot` z bootstrapem wolne na dużych danych | Wstępna agregacja w SQL/Polars/DuckDB i `errorbar=None`, lub zmniejsz `n_boot` |
| `swarmplot` wolny i ostrzega | Użyj `stripplot`, `boxenplot` lub `violinplot` |
| Rysowanie wielu wykresów w pętli zjada pamięć | `plt.close(fig)` po zapisie |
| Agregacje przed rysunkiem | DuckDB/Polars: `df_pl.group_by(...).agg(...).to_pandas()` → seaborn rysuje już tylko agregat |

### 14.3. Typowe pułapki

1. **Figure-level vs axes-level:** `sns.relplot(..., ax=ax)` nie zadziała. Do własnych subplots użyj `scatterplot`/`lineplot`; `FacetGrid` zwraca `g.axes`, `g.figure`.
2. **Seaborn ≥ 0.13:** przekazanie `palette` bez `hue` generuje ostrzeżenie (`FutureWarning`). Przypisz `hue=x` i `legend=False`.
3. **`set_xticklabels` bez `set_xticks`** (lub `FixedLocator`) daje przesunięte etykiety.
4. **Mieszanie `tight_layout` i `constrained_layout`:** wybierz jedno.
5. **Zmienne kategoryczne jako liczby** (`satisfaction` 1–5) seaborn traktuje jako ciągłe. Dla `catplot`/`boxplot` po osi X zadziała, ale dla `hue` ustaw `.astype("category")` albo `palette` dyskretną.
6. **Kolejność kategorii:** `pd.Categorical(..., ordered=True)` lub `order=` (np. dni tygodnia, miesiące).
7. **Oś Y barplot od zera.** Skracanie osi słupków zniekształca porównania.
8. **Ucięte pasma w `lineplot`:** dla jednej obserwacji na punkt X nie ma przedziału ufności, a `errorbar` nic nie zmieni.
9. **KDE poza zakresem danych:** ustaw `cut=0` lub `clip=(min, max)` (np. wartości nie mogą być ujemne).
10. **Daty:** upewnij się, że kolumna jest `datetime64`, a nie `object`/`str`.
11. **Rozmiar:** `figsize` w matplotlib (cale), a `height`/`aspect` w figure-level; `dpi` wpływa na rozmiar wyświetlania w notebooku.

## 15. Ściągawka: jaki wykres do jakiego pytania

| Pytanie | Wykres (seaborn / matplotlib) | Uwagi |
|---|---|---|
| Jak zmienia się wartość w czasie? | `lineplot`, `ax.plot`, `stackplot` | Dodaj średnią kroczącą, pasma, zdarzenia (`axvline`) |
| Porównanie wartości między kategoriami | `barplot`, `countplot`, lollipop | Sortuj, oś od 0 |
| Skład całości | 100% stacked bar, (ew.) pie/donut ≤ 5 kat. | Pie tylko przy dużych różnicach |
| Jaki jest kształt rozkładu? | `histplot`, `kdeplot`, `displot` | Sprawdź kilka `bins`; `log_scale` dla skośnych |
| Porównanie rozkładów dwóch+ grup | `ecdfplot`, `violinplot`, `boxplot`, ridge | ECDF najuczciwsze; violin/ridge najbardziej poglądowe |
| Percentyle i ogony | `ecdfplot`, `boxenplot` | `complementary=True` dla „% powyżej progu" |
| Pokaż każdą obserwację w grupie | `stripplot`, `swarmplot` | Małe próby; łącz z box/point |
| Średnie z niepewnością (2 czynniki) | `pointplot` | Równoległe linie = brak interakcji |
| Zależność X–Y | `scatterplot`, `regplot`, `lmplot` | Alfa, regresja, kolor/rozmiar jako kolejne wymiary |
| Zależność X–Y przy dużych danych | `hexbin`, `histplot(x,y)`, `kdeplot(x,y)`, `jointplot(kind="hex")` | Zamiast scatter |
| Zależność + rozkłady brzegowe | `jointplot`, `JointGrid` | |
| Przegląd wielu zmiennych (EDA) | `pairplot`, `PairGrid`, heatmapa korelacji | Max ~6 zmiennych |
| Macierz wartości / korelacje | `heatmap` (`center=0`, `mask`) | Paleta rozbieżna dla korelacji |
| Grupy podobnych zmiennych/obiektów | `clustermap` | Skaluj dane (`z_score`) |
| Ten sam wykres per kategoria | `FacetGrid`, `relplot`/`catplot`/`displot` z `col=` | *Small multiples* |
| Diagnostyka regresji | `residplot` | Wzorzec reszt = problem z modelem |
| Dwie miary w różnych jednostkach | `twinx` | Oszczędnie |

### Najważniejsze zasady dobrej wizualizacji

1. **Jedno przesłanie na wykres:** tytuł mówi, co widać („Premium ma 2× wyższy przychód"), a nie tylko „Przychód wg segmentu".
2. **Kolor ma znaczenie:** szary dla kontekstu, jeden akcent dla tego, co ważne; ten sam kolor = ta sama kategoria w całym raporcie.
3. **Usuń szum:** zbędne ramki i siatka, legenda tylko gdy konieczna (lepiej etykieta bezpośrednio przy serii).
4. **Uczciwe skale:** słupki od zera; unikaj podwójnych osi, jeśli nie są konieczne.
5. **Pokaż niepewność** (CI, pasma), a nie tylko punktowe estymaty.
6. **Dostępność:** `sns.color_palette("colorblind")` lub palety `viridis`/`rocket`/`mako` (czytelne w odcieniach szarości).

## Co dalej (rozwój w kierunku Twojego stacku)

- **Interaktywność:** `plotly.express` (te same koncepcje, hover i zoom), przydatne do eksploracji, a w raportach Power BI zostają natywne wizualizacje.
- **Seaborn objects API** (`import seaborn.objects as so`): deklaratywna gramatyka grafiki (`so.Plot(df, x=, y=).add(so.Dot()).add(so.Line(), so.PolyFit())`). Warto znać, bo to kierunek rozwoju seaborn.
- **Mapy:** GeoPandas ma `gdf.plot(column=, cmap=, legend=True)`, a to ta sama składnia colormap, `ax`, `legend_kwds` co powyżej. Choropleth + `contextily` (podkład map).
- **Duże dane:** agregacja w DuckDB/Polars → `.to_pandas()` → seaborn. Wykres powinien dostawać już zagregowane dane.
- **Statystyka na wykresach:** `statsmodels` (`qqplot`, `plot_acf`) i `scipy.stats` (np. `probplot`) świetnie uzupełniają EDA (normalność, autokorelacja).